In [15]:
import ROOT

In [16]:
file_path = "../root_ML/merged_ML-600to800.root"
file = ROOT.TFile(file_path)
tree = file.Get("jetTree")

n_entries = tree.GetEntries()
print(f"Number of entries in the tree: {n_entries}")

Number of entries in the tree: 5737037


In [ ]:
hist_2D = ROOT.TH2F("hist_2D", "2D Histogram; qq_score; matched_score", 10, 0, 1, 10, 0, 1)
hist_2D_ch1 = ROOT.TH2F("hist_2D_purity", "2D Histogram (Purity); qq_score; matched_score", 10, 0, 1, 10, 0, 1)

Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas


In [18]:
for event in range(n_entries):
    tree.GetEntry(event)
    for jet_i in range(len(tree.jet_pt)):
        qq_score = tree.lund_ML_qq[jet_i]
        matched_score = tree.lund_ML_matched[jet_i]

        channel2 = tree.lund_secondary_idx_sd[jet_i]

        hist_2D.Fill(qq_score, matched_score)
        if channel2 == 1:
            hist_2D_ch1.Fill(qq_score, matched_score)

In [19]:
for x_bin in range(1, hist_2D.GetNbinsX() + 1):
    for y_bin in range(1, hist_2D.GetNbinsY() + 1):
        total = hist_2D.GetBinContent(x_bin, y_bin)
        purity = hist_2D_ch1.GetBinContent(x_bin, y_bin)
        
        if total > 0:  # Avoid division by zero
            hist_2D_ch1.SetBinContent(x_bin, y_bin, purity / total)
        else:
            hist_2D_ch1.SetBinContent(x_bin, y_bin, 0)  # Set purity to 0 if no entries

In [26]:
canvas = ROOT.TCanvas("canvas", "Canvas", 800, 600)
hist_2D.GetXaxis().SetTitle("qq_score")
hist_2D.GetYaxis().SetTitle("matched_score")

#Set axis ranges
hist_2D.GetXaxis().SetRangeUser(0, 1)
hist_2D.GetYaxis().SetRangeUser(0, 1)


hist_2D.SetTitle("2D Histogram of qq_score vs matched_score")
hist_2D.SetStats(0)  # Disable statistics box
hist_2D_ch1.Draw("COLZ")  # Draw purity histogram on top
hist_2D.Draw("TEXT SAME")
canvas.Draw()

Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas


In [21]:
hist_dpsi = ROOT.TH1F("hist_dpsi", "Histogram of dpsi; dpsi; Entries", 20, 0, 3.14)

In [22]:
min_qq_score = 0.6
min_matched_score = 0.9
for event in range(n_entries):
    tree.GetEntry(event)
    for jet_i in range(len(tree.jet_pt)):
        qq_score = tree.lund_ML_qq[jet_i]
        matched_score = tree.lund_ML_matched[jet_i]
        
        if qq_score > min_qq_score and matched_score > min_matched_score:
            dpsi = tree.lund_psi12_sd[jet_i]
            hist_dpsi.Fill(dpsi)

In [23]:
canvas_dpsi = ROOT.TCanvas("canvas_dpsi", "Canvas for dpsi", 800, 600)
hist_dpsi.SetTitle(f"Histogram of dpsi for qq_score > {min_qq_score} and matched_score > {min_matched_score}")
hist_dpsi.SetStats(0)  # Disable statistics box
hist_dpsi.GetXaxis().SetTitle("dpsi")
hist_dpsi.GetYaxis().SetTitle("Entries")
hist_dpsi.Draw()
canvas_dpsi.Draw()

Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas_dpsi


In [27]:
import numpy as np
#calculate <cos(2dpsi)> of hist_dpsi

cos_2dpsi_sum = 0
total_entries = 0

for x_bin in range(1, hist_dpsi.GetNbinsX() + 1):
    dpsi = hist_dpsi.GetBinCenter(x_bin)
    entries = hist_dpsi.GetBinContent(x_bin)
    cos_2dpsi_sum += entries * np.cos(2 * dpsi)
    total_entries += entries

print(f"<cos(2dpsi)> = {cos_2dpsi_sum / total_entries if total_entries > 0 else 0}")

<cos(2dpsi)> = -0.29751881183396217
